In [2]:
!pip install 'jax[cpu] >= 0.6.0, < 0.7.0'
# jax[cuda12]         # If you want to use a GPU with CUDA 12
!pip install 'numpy >= 1.26.4'
!pip install 'scipy >= 1.12.0'
!pip install attrs
!pip install optax
!pip install filelock
!pip install nbconvert
!pip install 'mpax >= 0.2.4'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 9.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 MB 9.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [jax]3/4 [jax]ib]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [optax]32m1/2 [optax]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 MB 10.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.232m0/5 [typing_extensions]
  Attempting uninstall: jaxlib━━━━━━━━━━━━━━ 0/5 [typing_extensions]
    Found existing installation: jaxlib 0.6.2 0/5 [typing_extensions]
    Uninstalling jaxlib-0.6.2:━━━━━━━━━━━━━━ 0/5 [typing_extensions]
    

In [1]:
import pandas as pd
import jax.numpy as jnp
from jax.experimental.sparse import BCOO
import scipy as sp

In [25]:
def convert_COO_file_to_jax(file_name, n_cols=None):
    df = pd.read_csv(file_name)

    # Extract the 'row', 'column', and 'value' columns
    rows = df['row'].values
    cols = df['col'].values
    values = df['data'].values

    # Stack the row and column indices to create the indices array
    indices = jnp.vstack((rows, cols)).T  # Shape: (nnz, 2)

    # Convert the values to a JAX array
    data = jnp.array(values)

    # Determine the shape of the matrix (assuming zero-based indexing)
    if n_cols is None:
        shape = (rows.max()+1, cols.max()+1)
    else:
        shape = (rows.max()+1, n_cols)

    # Create the BCOO matrix
    mat = BCOO((data, indices), shape=shape)
    return mat

In [12]:
def formulate_milp_sparse_inputs(dt_sparse, d_oar1_sparse, d_oar2_sparse, d_oar3_sparse, d_oar4_sparse, Tmax, Tmin, UD_oar1, UD_oar2, UD_oar3, UD_oar4, UV_oar1, UV_oar2, UV_oar3, UV_oar4, M):
    """
    Formulate the mixed-integer linear program (MILP) for the radiation therapy optimization problem,
    using sparse input matrices and outputting the constraint matrix A in JAX's BCOO sparse format.

    Parameters:
    - dt_sparse: scipy.sparse CSR matrix of shape (n_t, m), dose-influence matrix for target
    - d_oar1_sparse: scipy.sparse CSR matrix of shape (n_oar1, m), dose-influence matrix for OAR1
    - d_oar2_sparse: scipy.sparse CSR matrix of shape (n_oar2, m), dose-influence matrix for OAR2
    - d_oar3_sparse: scipy.sparse CSR matrix of shape (n_oar3, m), dose-influence matrix for OAR3
    - d_oar4_sparse: scipy.sparse CSR matrix of shape (n_oar4, m), dose-influence matrix for OAR4
    - Tmax: scalar, upper bound for dose of each voxel in PTV
    - Tmin: scalar, lower bound for dose of each voxel in PTV
    - UD_oar1: scalar, dose-volume constraint for OAR1
    - UD_oar2: scalar, dose-volume constraint for OAR2
    - UD_oar3: scalar, dose-volume constraint for OAR3
    - UD_oar4: scalar, dose-volume constraint for OAR4
    - UV_oar1: scalar, volume fraction constraint for OAR1
    - UV_oar2: scalar, volume fraction constraint for OAR2
    - UV_oar3: scalar, volume fraction constraint for OAR3
    - UV_oar4: scalar, volume fraction constraint for OAR4
    - M: scalar, a large number representing the maximum dose

    Returns:
    - w: numpy array, objective function coefficients
    - A: BCOO sparse matrix, constraint matrix in BCOO format
    - b: numpy array, right-hand side vector
    """

    n_t, m = dt_sparse.shape
    n_oar1, m1 = d_oar1_sparse.shape
    n_oar2, m2 = d_oar2_sparse.shape
    n_oar3, m3 = d_oar3_sparse.shape
    n_oar4, m4 = d_oar4_sparse.shape

    assert m == m1 == m2 == m3 == m4, "Mismatch in number of beamlets"

    # Total number of variables
    n_vars = n_oar1 + n_oar2 + n_oar3 + n_oar4 + m + 2 * n_t  # y_oar1, y_oar2, y_oar3, y_oar4, x_j, sl_i, su_i

    # Total number of constraints
    n_constraints = 2 * n_t + n_oar1 + n_oar2 + n_oar3 + n_oar4 + 4  # Constraints 1-6

    # Initialize objective vector w
    w = np.zeros(n_vars)

    # Indices for variables
    idx_y_oar1 = np.arange(0, n_oar1)
    idx_y_oar2 = np.arange(n_oar1, n_oar1 + n_oar2)
    idx_y_oar3 = np.arange(n_oar1 + n_oar2, n_oar1 + n_oar2 + n_oar3)
    idx_y_oar4 = np.arange(n_oar1 + n_oar2 + n_oar3, n_oar1 + n_oar2 + n_oar3 + n_oar4)
    idx_x = np.arange(n_oar1 + n_oar2 + n_oar3 + n_oar4, n_oar1 + n_oar2 + n_oar3 + n_oar4 + m)
    idx_sl = np.arange(n_oar1 + n_oar2 + n_oar3 + n_oar4 + m, n_oar1 + n_oar2 + n_oar3 + n_oar4 + m + n_t)
    idx_su = np.arange(n_oar1 + n_oar2 + n_oar3 + n_oar4 + m + n_t, n_vars)

    # Set coefficients for sl_i and su_i in objective
    w[idx_sl] = 1
    w[idx_su] = 1

    # Prepare lists to construct sparse matrix
    data = []
    row_indices = []
    col_indices = []

    b = np.zeros(n_constraints)

    # Constraints 1: Upper bound on target dose
    dt_csr = dt_sparse.tocsr()
    for i in range(n_t):
        row = i
        # Get the indices and data for row i
        start = dt_csr.indptr[i]
        end = dt_csr.indptr[i+1]
        cols_i = dt_csr.indices[start:end]
        data_i = dt_csr.data[start:end]

        # Add entries for x_j variables
        data.extend(data_i)
        row_indices.extend([row]*len(cols_i))
        col_indices.extend(idx_x[cols_i])

        # - su_i
        data.append(-1)
        row_indices.append(row)
        col_indices.append(idx_su[i])

        b[row] = Tmax

    # Constraints 2: Lower bound on target dose
    for i in range(n_t):
        row = n_t + i
        start = dt_csr.indptr[i]
        end = dt_csr.indptr[i+1]
        cols_i = dt_csr.indices[start:end]
        data_i = dt_csr.data[start:end]

        data.extend(-data_i)
        row_indices.extend([row]*len(cols_i))
        col_indices.extend(idx_x[cols_i])

        # - sl_i
        data.append(-1)
        row_indices.append(row)
        col_indices.append(idx_sl[i])

        b[row] = -Tmin

    # Constraints 3: OAR1 dose limit
    d_oar1_csr = d_oar1_sparse.tocsr()
    for i in range(n_oar1):
        row = 2 * n_t + i
        start = d_oar1_csr.indptr[i]
        end = d_oar1_csr.indptr[i+1]
        cols_i = d_oar1_csr.indices[start:end]
        data_i = d_oar1_csr.data[start:end]

        data.extend(data_i)
        row_indices.extend([row]*len(cols_i))
        col_indices.extend(idx_x[cols_i])

        # - (M - UD_oar1) * y_oar1[i]
        data.append(-(M - UD_oar1))
        row_indices.append(row)
        col_indices.append(idx_y_oar1[i])

        b[row] = UD_oar1

    # Constraints 4: OAR2 dose limit
    d_oar2_csr = d_oar2_sparse.tocsr()
    for i in range(n_oar2):
        row = 2 * n_t + n_oar1 + i
        start = d_oar2_csr.indptr[i]
        end = d_oar2_csr.indptr[i+1]
        cols_i = d_oar2_csr.indices[start:end]
        data_i = d_oar2_csr.data[start:end]

        data.extend(data_i)
        row_indices.extend([row]*len(cols_i))
        col_indices.extend(idx_x[cols_i])

        # - (M - UD_oar2) * y_oar2[i]
        data.append(-(M - UD_oar2))
        row_indices.append(row)
        col_indices.append(idx_y_oar2[i])

        b[row] = UD_oar2
        
    # Constraints 5: OAR3 dose limit
    d_oar3_csr = d_oar3_sparse.tocsr()
    for i in range(n_oar3):
        row = 2 * n_t + n_oar1 + n_oar2 + i
        start = d_oar3_csr.indptr[i]
        end = d_oar3_csr.indptr[i+1]
        cols_i = d_oar3_csr.indices[start:end]
        data_i = d_oar3_csr.data[start:end]

        data.extend(data_i)
        row_indices.extend([row]*len(cols_i))
        col_indices.extend(idx_x[cols_i])

        # - (M - UD_oar3) * y_oar3[i]
        data.append(-(M - UD_oar3))
        row_indices.append(row)
        col_indices.append(idx_y_oar3[i])

        b[row] = UD_oar3
    
    # Constraints 6: OAR4 dose limit
    d_oar4_csr = d_oar4_sparse.tocsr()
    for i in range(n_oar4):
        row = 2 * n_t + n_oar1 + n_oar2 + n_oar3 + i
        start = d_oar4_csr.indptr[i]
        end = d_oar4_csr.indptr[i+1]
        cols_i = d_oar4_csr.indices[start:end]
        data_i = d_oar4_csr.data[start:end]

        data.extend(data_i)
        row_indices.extend([row]*len(cols_i))
        col_indices.extend(idx_x[cols_i])

        # - (M - UD_oar4) * y_oar4[i]
        data.append(-(M - UD_oar4))
        row_indices.append(row)
        col_indices.append(idx_y_oar4[i])

        b[row] = UD_oar4



    # Constraint 7: OAR1 voxel limit
    row = 2 * n_t + n_oar1 + n_oar2 + n_oar3 + n_oar4
    indices = idx_y_oar1
    values = np.ones(len(idx_y_oar1))

    data.extend(values)
    row_indices.extend([row]*len(indices))
    col_indices.extend(indices)

    b[row] = UV_oar1 * n_oar1

    # Constraint 8: OAR2 voxel limit
    row = 2 * n_t + n_oar1 + n_oar2 + n_oar3 + n_oar4 + 1
    indices = idx_y_oar2
    values = np.ones(len(idx_y_oar2))

    data.extend(values)
    row_indices.extend([row]*len(indices))
    col_indices.extend(indices)

    b[row] = UV_oar2 * n_oar2

    # Constraint 9: OAR3 voxel limit
    row = 2 * n_t + n_oar1 + n_oar2 + n_oar3 + n_oar4 + 2
    indices = idx_y_oar3
    values = np.ones(len(idx_y_oar3))

    data.extend(values)
    row_indices.extend([row]*len(indices))
    col_indices.extend(indices)

    b[row] = UV_oar3 * n_oar3

    # Constraint 10: OAR4 voxel limit
    row = 2 * n_t + n_oar1 + n_oar2 + n_oar3 + n_oar4 + 3
    indices = idx_y_oar4
    values = np.ones(len(idx_y_oar4))

    data.extend(values)
    row_indices.extend([row]*len(indices))
    col_indices.extend(indices)

    b[row] = UV_oar4 * n_oar4

    # Convert lists to JAX arrays
    data = jnp.array(data)
    row_indices = jnp.array(row_indices)
    col_indices = jnp.array(col_indices)
    shape = (n_constraints, n_vars)

    # Stack row and column indices to form the coordinates
    coords = jnp.stack((row_indices, col_indices), axis=-1)

    # Create BCOO sparse matrix
    A_sparse = BCOO((data, coords), shape=shape)

    return w, A_sparse, b

In [27]:
rectum_mat = convert_COO_file_to_jax('Rectum.csv')
bladder_mat = convert_COO_file_to_jax('Bladder.csv')
bone_Lt_mat = convert_COO_file_to_jax('Bone_Lt.csv')
bone_Rt_mat = convert_COO_file_to_jax('Bone_Rt.csv')
prostate_mat = convert_COO_file_to_jax('Prostate.csv')
n_oar1 = rectum_mat.shape[0]
n_oar2 = bladder_mat.shape[0]
n_oar3 = bone_Lt_mat.shape[0]
n_oar4 = bone_Rt_mat.shape[0]

UV_oar1 = 0.2
UV_oar2 = 0.35
UV_oar3 = 0.03
UV_oar4 = 0.03

tmin = 171
tmax = 180
UD_oar1 = 170
UD_oar2 = 171
UD_oar3 = 122
UD_oar4 = 122
M = 200
Vt = prostate_mat.shape[0]
B = prostate_mat.shape[1]
r = 1
ub = 200

In [28]:
to_scipy = lambda x: sp.sparse.coo_array((x.data, x.indices.transpose()), shape=x.shape)
w, A, b = formulate_milp_sparse_inputs(to_scipy(prostate_mat), to_scipy(rectum_mat), to_scipy(bladder_mat), to_scipy(bone_Lt_mat), to_scipy(bone_Rt_mat), tmax, tmin, UD_oar1, UD_oar2, UD_oar3, UD_oar4,  UV_oar1, UV_oar2,UV_oar3, UV_oar4, M)